In [ ]:
# Imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

In [ ]:
# Carregar dados do banco
load_dotenv()

server = os.getenv('SQL_SERVER')
database = os.getenv('SQL_DATABASE')

connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

# Buscar dados combinando as tabelas clientes, solicitacoes e decisoes
# Filtra apenas creditos aprovados para treinar o modelo
query = """
    SELECT 
        clientes.idade,
        clientes.renda_mensal,
        clientes.score_credito,
        clientes.tem_imovel,
        clientes.tem_veiculo,
        clientes.tempo_emprego_anos,
        clientes.qtd_emprestimos_ativos,
        clientes.historico_inadimplencia,
        clientes.possui_restricao,
        clientes.historico_produto,
        solicitacoes_credito.valor_solicitado,
        solicitacoes_credito.prazo_meses,
        solicitacoes_credito.tipo_credito,
        solicitacoes_credito.nivel_risco,
        decisoes.inadimplente
    FROM solicitacoes_credito
    JOIN clientes ON solicitacoes_credito.id_cliente = clientes.id_cliente
    JOIN decisoes ON solicitacoes_credito.id_solicitacao = decisoes.id_solicitacao
    WHERE decisoes.resultado = 'aprovado'
"""

df = pd.read_sql(query, engine)

print(f"Total de registros carregados: {len(df)}")
print(df.head())

In [ ]:
# Preparar os dados para o modelo
# Converter True/False para 1/0
colunas_booleanas = ['tem_imovel', 'tem_veiculo', 'historico_inadimplencia', 'possui_restricao', 'inadimplente']
for coluna in colunas_booleanas:
    df[coluna] = df[coluna].astype(int)

# Converter colunas de texto em numeros pois o modelo so trabalha com numeros
encoder = LabelEncoder()
df['historico_produto'] = df['historico_produto'].fillna('sem_historico')
df['historico_produto'] = encoder.fit_transform(df['historico_produto'])
df['tipo_credito'] = encoder.fit_transform(df['tipo_credito'])
df['nivel_risco'] = encoder.fit_transform(df['nivel_risco'])

# Separar as features (X) do target (y)
X = df.drop(columns=['inadimplente'])
y = df['inadimplente']

# Dividir em treino (80%) e teste (20%)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# Balancear as classes com SMOTE
# SMOTE cria exemplos sinteticos de inadimplentes para equilibrar os dados
smote = SMOTE(random_state=42)
X_treino, y_treino = smote.fit_resample(X_treino, y_treino)

print(f"Registros para treino apos balanceamento: {len(X_treino)}")
print(f"Registros para teste: {len(X_teste)}")
print(f"Distribuicao do treino: {y_treino.value_counts().to_dict()}")

In [ ]:
# Treinar o modelo Random Forest com peso maior para inadimplentes
modelo = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
modelo.fit(X_treino, y_treino)

print("Modelo treinado com sucesso!")

In [ ]:
# Avaliar o desempenho do modelo
y_previsao = modelo.predict(X_teste)

print(f"Acuracia: {accuracy_score(y_teste, y_previsao):.2%}")
print("\nRelatorio completo:")
print(classification_report(y_teste, y_previsao))